<a href="https://colab.research.google.com/github/darksister1331/ActionBoard-AI/blob/main/Live_Transcript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#%% Imports
import os
import time
import subprocess
import signal
from datetime import datetime
import threading
import pygetwindow as gw
import speech_recognition as sr
import pyaudio

#%% CONFIGURATION
ZOOM_LINK = "zoom_link"
AUDIO_DEVICE_NAME = "Microphone Array (Intel® Smart Sound Technology for Digital Microphones)"
FFMPEG_PATH = r"File path for location of ffmpeg.exe"

# You can also use the function below to get the mic index:
def get_device_index(name_fragment):
    pa = pyaudio.PyAudio()
    for i in range(pa.get_device_count()):
        info = pa.get_device_info_by_index(i)
        if name_fragment.lower() in info["name"].lower():
            return i
    return None

MIC_INDEX = get_device_index("Microphone Array")

#%% FILE SETUP
recordings_dir = "Recordings"
os.makedirs(recordings_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
mkv_file = os.path.join(recordings_dir, f"zoom_meeting_{timestamp}.mkv")
mp4_file = os.path.join(recordings_dir, f"zoom_meeting_{timestamp}.mp4")
audio_file = os.path.join(recordings_dir, f"zoom_meeting_{timestamp}.mp3")
transcript_file = os.path.join(recordings_dir, f"transcript_{timestamp}.txt")

#%% Function: Check if Zoom window is open
def is_zoom_meeting_window_open():
    return any("zoom meeting" in title.lower() for title in gw.getAllTitles())

#%% Shared stop flag for transcription
from threading import Event
stop_transcription_event = Event()

#%% Function: Live transcriber
def run_transcriber():
    recognizer = sr.Recognizer()
    mic = sr.Microphone(device_index=MIC_INDEX)
    with mic as source, open(transcript_file, "a", encoding="utf-8") as f:
        recognizer.adjust_for_ambient_noise(source)
        print(" Live transcription started...")
        try:
            while not stop_transcription_event.is_set():
                audio = recognizer.listen(source, phrase_time_limit=5)
                try:
                    text = recognizer.recognize_google(audio)
                    if len(text.strip()) > 2 and text.isascii():
                        print("📝", text)
                        f.write(text + "\n")
                        f.flush()
                except sr.UnknownValueError:
                    print("Could not understand")
                except sr.RequestError as e:
                    print(f" API error: {e}")
        except Exception as e:
            print(" Transcriber error:", e)

#%% Step 1: Launch Zoom
print(" Opening Zoom...")
os.startfile(ZOOM_LINK)
time.sleep(15)

#%% Step 2: Start FFmpeg Recording
print(" Starting FFmpeg screen + audio recording...")
ffmpeg_command = [
    FFMPEG_PATH,
    "-y",
    "-f", "gdigrab",
    "-framerate", "30",
    "-i", "desktop",
    "-f", "dshow",
    "-i", f"audio={AUDIO_DEVICE_NAME}",
    "-vcodec", "libx264",
    "-acodec", "aac",
    "-pix_fmt", "yuv420p",
    "-preset", "ultrafast",
    mkv_file
]
recording_process = subprocess.Popen(ffmpeg_command, creationflags=subprocess.CREATE_NEW_PROCESS_GROUP)

#%% Step 3: Start Transcription
transcriber_thread = threading.Thread(target=run_transcriber, daemon=True)
transcriber_thread.start()

#%% Step 4: Monitor Zoom Meeting
print(" Monitoring Zoom window...")
try:
    while True:
        if not is_zoom_meeting_window_open():
            print(" Zoom meeting ended.")
            stop_transcription_event.set()
            break
        time.sleep(5)
except KeyboardInterrupt:
    print(" Stopped manually.")
    stop_transcription_event.set()

#%% Step 5: Stop Recording
recording_process.send_signal(signal.CTRL_BREAK_EVENT)
try:
    recording_process.wait(timeout=10)
except subprocess.TimeoutExpired:
    recording_process.kill()
transcriber_thread.join()
print(f" MKV recording saved: {os.path.abspath(mkv_file)}")

#%% Step 6: Convert to MP4
print(" Converting MKV to MP4...")
subprocess.run([
    FFMPEG_PATH,
    "-i", mkv_file,
    "-c", "copy",
    "-movflags", "+faststart",
    mp4_file
])
print(f" MP4 saved: {os.path.abspath(mp4_file)}")

#%% Step 7: Extract MP3
print(" Extracting audio to MP3...")
subprocess.run([
    FFMPEG_PATH,
    "-i", mkv_file,
    "-vn",
    "-acodec", "mp3",
    audio_file
])
print(f" MP3 saved: {os.path.abspath(audio_file)}")
print(f" Transcript saved: {os.path.abspath(transcript_file)}")
